In [ ]:
# Tworzenie bazowego modelu ONNX

import torch

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch

import torch
from model import ResNet

# 1. Load the trained PyTorch model
net = ResNet(3, 11)
net.cpu()

# Nie możemy użyć kompilowanej sieci do eksportu do ONNX.
# Całe szczęście, możemy uzyskać oryginalny model przed kompilacją usuwając prefiksy z nazw warstw.
# net = torch.compile(net)

state_dict = torch.load('best.pth', map_location='cpu')

# Usuwanie prefiksów '_orig_mod.' lub innych technicznych nazw
new_state_dict = {}
for k, v in state_dict.items():
    name = k.replace('_orig_mod.', '') # usuwa prefiks kompilatora
    new_state_dict[name] = v

net.load_state_dict(new_state_dict)
# Konieczne jest ustawienie modelu w tryb ewaluacji przed eksportem
# żeby BatchNorm i Dropout działały poprawnie
net.eval()

# 2. Przykładowe dane wejściowe modelu
dummy_input = torch.randn(32, 3, 256, 256)

# 3. Eksport z dynamicznymi rozmiarami wejścia
torch.onnx.export(
    net,
    dummy_input,
    "model.onnx",
    export_params=True,  
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'}, # Potrzebne dla różnych rozmiarów batcha
        'output': {0: 'batch_size'}
    }
)

/tmp/ipykernel_17045/2713775826.py:35: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 23 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.9.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[s77,3,256,256]>
            ),
            outputs=(
                %"output"<FLOAT,[s77,11]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[128,3,3,3]>{Tensor(...)},
                %"res_layer_1.0.conv1.weight"<FLOAT,[256,128,3,3]>{Tensor(...)},
                %"res_layer_1.0.conv2.weight"<FLOAT,[256,256,3,3]>{Tensor(...)},
                %"res_layer_1.0.shortcut.0.weight"<FLOAT,[256,128,1,1]>{Tensor(...)},
                %"res_layer_1.1.conv1.weight"<FLOAT,[256,256,3,3]>{Tensor(...)},
                %"res_layer_1.1.conv2.weight"<FLOAT,[256,256,3,3]>{Tensor(...)},
                %"res_layer_2.0.conv1.weight"<

In [ ]:
# Tworzenie modelu uwzględniającego transformacje wstępne

import torch
import torch.nn as nn
import torch.nn.functional as F
from test_transforms import ManualTransform

net_with_transforms = nn.Sequential(ManualTransform(), net)
net_with_transforms.eval()

dummy_input = torch.randn(32, 3, 256, 256)

torch.onnx.export(
    net_with_transforms,
    dummy_input,
    "model_with_transforms.onnx",
    export_params=True,        
    do_constant_folding=True,  
    input_names=['input'],     
    output_names=['output'],   
    dynamic_axes={
        'input': {0: 'batch_size', 2: 'height', 3: 'width'}, # Różne rozmiary batcha, wysokości i szerokości
        'output': {0: 'batch_size'}
    }
)

import onnx

onnx_model = onnx.load("model_with_transforms.onnx")

onnx.checker.check_model(onnx_model)

print("Model checked successfully!")

Sequential(
  (0): ManualTransform()
  (1): ResNet(
    (conv1): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (drop): Dropout(p=0.5, inplace=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (res_layer_1): Sequential(
      (0): ResidualBlock(
        (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Sequential(
          (0): Conv2d(128, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): ResidualBlock(
        (conv1): Conv2d(256, 

In [ ]:
# Testowanie modelu ONNX z transformacjami wstępnymi

import numpy as np
from PIL import Image
import torchvision
import onnxruntime as ort

class NumpyToTensor:
    """
    A callable class to convert a PIL Image to a NumPy array in C,H,W format
    and scale it to [0.0, 1.0], mimicking torchvision.transforms.ToTensor().
    """
    def __call__(self, pil_img: Image.Image) -> np.ndarray:
        # Convert PIL image to numpy array, ensuring it's float32
        img_array = np.array(pil_img, dtype=np.float32)

        if len(img_array.shape) == 2:
            img_array = np.expand_dims(img_array, axis=-1)
            img_array = np.repeat(img_array, 3, axis=-1)

        img_array = np.transpose(img_array, (2, 0, 1))
        
        return img_array / 255.0


weather_images = torchvision.datasets.ImageFolder(root='dataset', transform=NumpyToTensor())
loader = torch.utils.data.DataLoader(weather_images, batch_size=1, shuffle=True, num_workers=4)
classes = weather_images.classes

sess_options = ort.SessionOptions()
# automatyczna optymalizacja grafu ONNX
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
# sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL # Zwykle szybsze dla pojedynczego strumienia

# Intel ONNX Runtime z OpenVINO - Optymalizacja pod inference na CPU
session = ort.InferenceSession("model_with_transforms.onnx", sess_options, providers=['OpenVINOExecutionProvider', 'CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}
total = 0
correct = 0
with torch.no_grad():
    for data in loader:
        images, labels = data
        
        input_data = images.numpy().astype(np.float32)
        outputs = session.run([output_name], {input_name: input_data})

        logits = outputs[0]
        predictions = np.argmax(logits, axis=1)

        for label, pred in zip(labels.numpy(), predictions):
            if label == pred:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1
            
            # Global counters
            total += 1
            if label == pred:
                correct += 1
          
        print(f'Processed {total} images so far...')
        if total > 100:
            break

        # print accuracy for each class
        for classname, correct_count in correct_pred.items():
            if (total_pred[classname] > 0):
                accuracy = 100 * float(correct_count) / total_pred[classname]
                print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')
        print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

    # Use float division for accuracy to avoid 0% due to integer floor division
    overall_acc = 100 * correct / total
    print(f'Final accuracy: {overall_acc:.2f} %')


Processed 1 images so far...
Accuracy for class: glaze is 100.0 %
Accuracy of the network on the 10000 test images: 100 %
Processed 2 images so far...
Accuracy for class: dew   is 100.0 %
Accuracy for class: glaze is 100.0 %
Accuracy of the network on the 10000 test images: 100 %
Processed 3 images so far...
Accuracy for class: dew   is 100.0 %
Accuracy for class: frost is 100.0 %
Accuracy for class: glaze is 100.0 %
Accuracy of the network on the 10000 test images: 100 %
Processed 4 images so far...
Accuracy for class: dew   is 100.0 %
Accuracy for class: frost is 100.0 %
Accuracy for class: glaze is 100.0 %
Accuracy of the network on the 10000 test images: 100 %


KeyboardInterrupt: 